# 15 - Google Places Feasibility

This notebook tests whether Google Places (New) can improve naming and identity for the most important Istanbul POIs.

Purpose:
- query Google Places only for the top important POIs
- compare returned place names with current OSM / `name_en` values
- decide whether Google Places is worth using as a naming / identity enrichment layer

Important:
- This notebook is **not** for replacing OSM.
- It is only for testing whether Google Places improves naming / identity for important landmarks.

In [37]:
import os
import time
import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load enriched POIs

In [38]:
pois = pd.read_csv("../data/processed/poi_enriched.csv")
pois.head()

,poi_id,name,name_en,category,category_clean,lat,lon,distance_to_center_km,nearby_count_500m,cluster_id,category_score,landmark_name_score,centrality_score,density_score,importance_score,is_park,is_historic,is_museum,is_attraction,is_religious
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007586,28.975554,0.248384,64,12,0.82,0.55,0.997367,0.941176,0.863004,0,1,0,0,0
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007289,28.975329,0.276907,63,12,0.82,0.55,0.997021,0.926471,0.859224,0,1,0,0,0
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic:castle,historic,41.006397,28.974808,0.361956,63,12,0.82,0.55,0.995990,0.926471,0.858915,0,1,0,0,0
3,311681431,Sağlık Müzesi,NaN,tourism:museum,museum,41.008314,28.975290,0.261290,66,12,0.85,0.35,0.997210,0.970588,0.849310,0,0,1,0,0
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,tourism:museum,museum,41.004295,28.977433,0.441766,55,12,0.85,0.59,0.995023,0.808824,0.844213,0,0,1,0,0


## Select the important landmark subset

In [39]:
TARGET_CATEGORIES = ["museum", "historic", "attraction"]
TOP_N = 3

test_pois = (
    pois[pois["category_clean"].isin(TARGET_CATEGORIES)]
    .sort_values("importance_score", ascending=False)
    .head(TOP_N)
    .copy()
)

test_pois["match_name"] = test_pois["name_en"].fillna(test_pois["name"])
test_pois[["poi_id", "name", "name_en", "match_name", "category_clean", "importance_score"]].head(20)


,poi_id,name,name_en,match_name,category_clean,importance_score
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,historic,0.863004
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,Antiochos Sarayı'nın Kalıntıları,historic,0.859224
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,historic,0.858915


## Google Places configuration

According to the official Google Places (New) docs:
- Text Search uses `POST https://places.googleapis.com/v1/places:searchText`
- requests use the `X-Goog-Api-Key` header
- responses are controlled with the `X-Goog-FieldMask` header

Set your key in the environment before running:

```bash
export GOOGLE_PLACES_API_KEY="..."
```

In [40]:
import os

GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_PLACES_API_KEY")
if not GOOGLE_PLACES_API_KEY:
    GOOGLE_PLACES_API_KEY = input("Paste Google Places API key: " ).strip()

GOOGLE_TEXT_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"
print("Key loaded:", bool(GOOGLE_PLACES_API_KEY))


Key loaded: True


## Helper function

In [41]:
def search_google_place(query, lat, lon):
    if not GOOGLE_PLACES_API_KEY:
        raise ValueError("GOOGLE_PLACES_API_KEY is missing")

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_PLACES_API_KEY,
        "X-Goog-FieldMask": ",".join([
            "places.id",
            "places.displayName",
            "places.formattedAddress",
            "places.primaryTypeDisplayName",
            "places.location",
            "places.rating",
        ]),
    }

    payload = {
        "textQuery": f"{query}, Istanbul",
        "locationBias": {
            "circle": {
                "center": {
                    "latitude": float(lat),
                    "longitude": float(lon),
                },
                "radius": 500.0,
            }
        }
    }

    response = requests.post(GOOGLE_TEXT_SEARCH_URL, headers=headers, json=payload, timeout=20)

    if response.status_code != 200:
        debug = {
            'status_code': response.status_code,
            'response_text': response.text[:500],
            'x_ratelimit_limit': response.headers.get('X-RateLimit-Limit'),
            'x_ratelimit_remaining': response.headers.get('X-RateLimit-Remaining'),
            'retry_after': response.headers.get('Retry-After'),
        }
        print("Google Places error debug:", debug)
        raise RuntimeError(f"Google Places request failed: {debug}")

    return response.json()


## Test a few landmark queries first

In [42]:
first_row = test_pois[["match_name", "lat", "lon"]].iloc[0]
print(f"Testing one query only: {first_row['match_name']}")

if not GOOGLE_PLACES_API_KEY:
    print("Skipping because GOOGLE_PLACES_API_KEY is missing")
else:
    try:
        result = search_google_place(first_row["match_name"], first_row["lat"], first_row["lon"])
        for item in result.get("places", [])[:3]:
            print(item.get("displayName", {}).get("text"), "|", item.get("id"), "|", item.get("formattedAddress"))
    except Exception as e:
        print("ERROR:", e)


Testing one query only: Lausos Sarayı'nın Kalıntıları
Google Places error debug: {'status_code': 403, 'response_text': '{\n  "error": {\n    "code": 403,\n    "message": "Places API (New) has not been used in project 1014682454971 before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/places.googleapis.com/overview?project=1014682454971 then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry.",\n    "status": "PERMISSION_DENIED",\n    "details": [\n      {\n        "@type": "type.googleapis.com/google.rpc.ErrorInfo",\n  ', 'x_ratelimit_limit': None, 'x_ratelimit_remaining': None, 'retry_after': None}
ERROR: Google Places request failed: {'status_code': 403, 'response_text': '{\n  "error": {\n    "code": 403,\n    "message": "Places API (New) has not been used in project 1014682454971 before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/places.g

## Run the top-POI Google Places name test

In [43]:
google_rows = []
stop_batch = False

for _, row in test_pois.iterrows():
    if stop_batch:
        break

    query = row["match_name"]

    record = {
        "poi_id": row["poi_id"],
        "osm_name": row["name"],
        "osm_name_en": row["name_en"],
        "match_name": query,
        "category_clean": row["category_clean"],
        "importance_score": row["importance_score"],
        "google_found": False,
        "google_display_name": None,
        "google_place_id": None,
        "google_formatted_address": None,
        "google_primary_type": None,
        "google_rating": None,
        "notes": None,
    }

    if not GOOGLE_PLACES_API_KEY:
        google_rows.append(record)
        continue

    try:
        result = search_google_place(query, row["lat"], row["lon"])
        places = result.get("places", [])

        if places:
            top = places[0]
            record.update({
                "google_found": True,
                "google_display_name": top.get("displayName", {}).get("text"),
                "google_place_id": top.get("id"),
                "google_formatted_address": top.get("formattedAddress"),
                "google_primary_type": top.get("primaryTypeDisplayName", {}).get("text"),
                "google_rating": top.get("rating"),
            })
    except Exception as e:
        record["notes"] = str(e)
        err = str(e)
        if any(k in err for k in ["PERMISSION_DENIED", "403", "429", "quota", "disabled"]):
            print("Stopping batch early to avoid wasting credits/calls.")
            stop_batch = True

    google_rows.append(record)
    if not stop_batch:
        time.sleep(1.0)

google_results = pd.DataFrame(google_rows)
google_results.head(20)


Google Places error debug: {'status_code': 403, 'response_text': '{\n  "error": {\n    "code": 403,\n    "message": "Places API (New) has not been used in project 1014682454971 before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/places.googleapis.com/overview?project=1014682454971 then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry.",\n    "status": "PERMISSION_DENIED",\n    "details": [\n      {\n        "@type": "type.googleapis.com/google.rpc.ErrorInfo",\n  ', 'x_ratelimit_limit': None, 'x_ratelimit_remaining': None, 'retry_after': None}
Stopping batch early to avoid wasting credits/calls.


,poi_id,osm_name,osm_name_en,match_name,category_clean,importance_score,google_found,google_display_name,google_place_id,google_formatted_address,google_primary_type,google_rating,notes
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,historic,0.863004,False,None,None,None,None,None,Google Places request failed: {'status_code': ...


## Manual evaluation fields

In [44]:
google_results["clearly_better_name"] = ""
google_results["would_use_for_wiki"] = ""
google_results["would_use_for_besttime"] = ""

google_results[[
    "poi_id",
    "osm_name",
    "osm_name_en",
    "match_name",
    "google_found",
    "google_display_name",
    "google_primary_type",
    "google_formatted_address",
    "clearly_better_name",
    "would_use_for_wiki",
    "would_use_for_besttime",
    "notes",
]].head(30)

,poi_id,osm_name,osm_name_en,match_name,google_found,google_display_name,google_primary_type,google_formatted_address,clearly_better_name,would_use_for_wiki,would_use_for_besttime,notes
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,False,None,None,None,,,,Google Places request failed: {'status_code': ...


## Save results for review

In [45]:
if not GOOGLE_PLACES_API_KEY:
    print("Skipping save because no Google Places API key was provided.")
elif 'google_results' not in globals() or google_results.empty:
    print("Skipping save because there are no Google results yet.")
elif not google_results['google_found'].any():
    print("Skipping save because no successful Google matches were returned.")
else:
    google_results.to_csv("../data/processed/google_places_name_test_top30.csv", index=False)
    print("Saved: ../data/processed/google_places_name_test_top30.csv")
    print("Shape:", google_results.shape)


Skipping save because no successful Google matches were returned.


## Decision rule

Use Google Places only if:
- setup is easy,
- names are clearly better for a majority of the tested landmarks,
- and it directly improves Wikipedia / BestTime query naming.

Otherwise, keep OSM + manually cleaned English names and move on.